# 04 — Backtest Results

The study's **authoritative results**, across all **10 pairs**. Everything below is read
from `reports/results/metrics.csv` and `run_manifest.json`, which `make run` produced from
`configs/pairs.yaml` — so nothing here can quietly disagree with what the pipeline
actually computed. The figures are regenerated live through
`pairs_teardown.study.run_study`, the same function the script calls.

This notebook deliberately contains **no strategy logic**. Where you see a hedge ratio or
a Sharpe, it came out of the package.

**One flat universe.** Every pair was specified from economic reasoning before it was run,
and every pair is reported here whatever it did. There is no headline set and no appendix
set — the config schema has no field that could express one, and `to_frame` emits no column
that could rank them. §5 explains why that constraint is doing real work in this
particular study.

| Principle | What it demands | Where |
|---|---|---|
| 4 — gross vs net | every result stated before *and* after transaction costs | §3, §4 |
| 5 — IS vs OOS | parameters fit in-sample; out-of-sample scored once, reported as-is | §2, §5 |

**Where the rest of the argument lives.** This notebook reports what the frozen
configuration produced, and nothing else. Every "but what if you had chosen differently"
question — window, entry/exit bands, cost level, split date, hedge specification — is
answered in `04_sensitivity_analysis.ipynb`, deliberately kept separate so that results and
robustness checks cannot be confused for one another. The narrative and conclusions are in
`05_writeup.ipynb`.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

from pairs_teardown.config import load_config
from pairs_teardown.data.loaders import load_or_download
from pairs_teardown.plotting.charts import plot_drawdown, plot_equity_curve
from pairs_teardown.study import run_study

# Notebooks run from notebooks/; every path below is anchored to the repo root so the
# notebook behaves identically however the kernel was launched.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

cfg = load_config(ROOT / "configs" / "pairs.yaml")
metrics = pd.read_csv(ROOT / "reports" / "results" / "metrics.csv")
manifest = json.loads((ROOT / "reports" / "results" / "run_manifest.json").read_text())

PAIRS = [p.name for p in cfg.pairs]

pd.set_option("display.precision", 4)
print(f"{len(PAIRS)} pairs, {len(metrics)} metric rows\n")
for p in cfg.pairs:
    print(f"  {p.name:<10} {p.a:<5} / {p.b:<5}  {p.rationale}")


## 1. Run provenance

What was run, with which parameters. `window`, `entry` and `exit` are **pre-registered
and frozen** — chosen a priori (60 days ≈ a trading quarter; 2.0/0.5 are the textbook
bands), never selected by looking at a return. What the study would have reported under
other choices is the subject of `04_sensitivity_analysis.ipynb`; the answer is unflattering
enough that it belongs in the conclusion, not in a footnote.


In [ ]:
frozen = ["window", "entry", "exit", "signal_hedge", "sizing_hedge"]
costs = ["commission_bps", "slippage_bps"]
sample = ["data_start", "in_sample_end", "data_end"]

for label, keys in [("FROZEN SIGNAL", frozen), ("COSTS", costs), ("SAMPLE", sample)]:
    print(label)
    for k in keys:
        print(f"  {k:<16} {manifest[k]}")

print("\nStatic SIZING hedge ratios (fit on in-sample data ONLY, then frozen):")
for name, g in manifest["hedge_ratios"].items():
    print(f"  {name:<10} g = {g:7.4f}")

print(f"\nrun at {manifest['run_utc']}")


## 2. Headline — all 10 pairs

Net total return, in-sample versus out-of-sample, sorted by the out-of-sample column.


In [ ]:
def wide(basis, value):
    v = metrics[(metrics.basis == basis) & (metrics.period != "full")]
    return v.pivot_table(index="pair", columns="period", values=value)[
        ["in_sample", "out_of_sample"]
    ]


net_ret = wide("net", "total_return") * 100
headline = pd.concat(
    {
        "net total return %": net_ret,
        "net Sharpe": wide("net", "sharpe"),
    },
    axis=1,
).sort_values(("net total return %", "out_of_sample"), ascending=False)
headline.round(2)


**Four of ten pairs are profitable out-of-sample after costs; six are not.** The range
runs from **UPS/FDX at −42.4%** to **UNP/CSX at +55.9%** — a spread of nearly 100
percentage points across ten pairs that were all chosen on the same kind of economic
reasoning, all traded with identical frozen parameters, over the same three years.

That dispersion, not the sign of any individual pair, is the study's central result. §5
develops it.


## 3. Gross versus net (principle 4)

The comparison most published pairs-trading backtests omit. `drag` is the annualized
return the strategy hands to the broker.


In [ ]:
rows = []
for period in ["in_sample", "out_of_sample"]:
    for pair in PAIRS:
        sel = metrics[(metrics.pair == pair) & (metrics.period == period)]
        g = sel[sel.basis == "gross"].iloc[0]
        n = sel[sel.basis == "net"].iloc[0]
        rows.append({
            "pair": pair,
            "period": period,
            "gross ann %": g.annualized_return * 100,
            "net ann %": n.annualized_return * 100,
            "drag %": (g.annualized_return - n.annualized_return) * 100,
            "gross Sharpe": g.sharpe,
            "net Sharpe": n.sharpe,
            "trades": int(g.n_trades),
        })

gross_net = pd.DataFrame(rows).set_index(["period", "pair"])
gross_net.round(2)


In [ ]:
oos = metrics[metrics.period == "out_of_sample"]
gross_oos = oos[oos.basis == "gross"].set_index("pair").total_return * 100
net_oos = oos[oos.basis == "net"].set_index("pair").total_return * 100

flipped = sorted(gross_oos[(gross_oos > 0) & (net_oos < 0)].index)
print(f"mean OOS total return    gross {gross_oos.mean():6.2f}%   ->   net {net_oos.mean():6.2f}%")
print(f"costs consume {(gross_oos.mean() - net_oos.mean()) / gross_oos.mean():.0%} of the mean gross return")
print(f"\npairs whose SIGN is decided by costs (gross > 0, net < 0): {flipped}")


Two things costs do here, which are worth separating because the earlier six-pair
version of this study conflated them:

**Costs consume ~80% of the average gross return.** The cross-sectional mean goes from
+3.1% to +0.6% over three years. That is the honest size of the friction effect.

**But costs no longer decide every sign.** They flip only two pairs (FOXA/FOX and MA/V)
from marginally positive to marginally negative. UNP/CSX and DUK/SO clear their costs
comfortably; UPS/FDX and XOM/CVX were losing badly before costs were charged. Attributing
the negative pairs to transaction costs would be wrong — most of them simply had no gross
edge.

**SPY/VOO remains the extreme case**: an out-of-sample gross Sharpe near zero becomes
**−2.36** net. The two ETFs track the same index, so the spread is almost perfectly tight —
a tiny edge, a tinier standard deviation, and a cost charge that does not shrink along with
them. A near-degenerate pair does not have a small cost problem; it has the worst one.


## 4. Where the money goes

If the cost story is real, the drag should be predictable from first principles rather
than being an unexplained residual. One unit of spread is long \$1 of A and short \$g of
B, so a unit change in position trades `(1 + |g|)` of notional and is charged commission +
slippage on it:

$$\text{annual drag} \;\approx\; \underbrace{\text{turnover}}_{\text{one-way, annualized}} \times\; (1 + |g|) \;\times\; \text{(commission + slippage)}$$


In [ ]:
bps = (manifest["commission_bps"] + manifest["slippage_bps"]) / 1e4

rows = []
for pair in PAIRS:
    sel = metrics[(metrics.pair == pair) & (metrics.period == "out_of_sample")]
    g = sel[sel.basis == "gross"].iloc[0]
    n = sel[sel.basis == "net"].iloc[0]
    actual = g.annualized_return - n.annualized_return
    predicted = g.turnover * (1 + abs(g.hedge_ratio)) * bps
    rows.append({
        "pair": pair,
        "turnover": g.turnover,
        "1+|g|": 1 + abs(g.hedge_ratio),
        "predicted drag %": predicted * 100,
        "actual drag %": actual * 100,
        "actual / predicted": actual / predicted,
    })

recon = pd.DataFrame(rows).set_index("pair")
display(recon.round(3))
print(f"actual/predicted ranges {recon['actual / predicted'].min():.3f} to "
      f"{recon['actual / predicted'].max():.3f} across all {len(PAIRS)} pairs")


The ratio lands between **0.84 and 1.16** across the ten pairs, centred on 1. The
residual scatter is expected rather than unexplained: the prediction is a simple
per-unit-turnover charge, while the actual figure is a difference of two *annualized
compound* returns over a three-year window, so a pair whose costs land unevenly against its
gains — UNP/CSX at 1.16, UPS/FDX at 0.84 — separates the two slightly. Over the longer
in-sample window the same check came in at 0.966–0.998.

The point stands: the cost model is doing what its docstring says and nothing else, no
hidden charge and no missing one. That closes off the most common objection to this kind of
result ("the cost assumption must be too harsh"). The assumption is 1 bp commission + 5 bps
slippage per side, which is not aggressive for liquid US large caps, and the drag it
produces is explained by how often the strategy trades.


## 5. The result: dispersion, not direction

Ten pairs, one set of frozen parameters, one out-of-sample period. If the strategy had a
real and general edge, the ten outcomes should cluster somewhere above zero. If it had
none, they should cluster at or below zero after costs. Neither happens.


In [ ]:
ins = net_ret["in_sample"]
oos_net = net_ret["out_of_sample"]

t_stat, p_val = stats.ttest_1samp(oos_net, 0.0)

print("OUT-OF-SAMPLE net total return, cross-section of 10 pairs")
print(f"  mean      {oos_net.mean():7.2f}%")
print(f"  median    {oos_net.median():7.2f}%")
print(f"  std dev   {oos_net.std():7.2f}%")
print(f"  min / max {oos_net.min():7.1f}% / {oos_net.max():.1f}%")
print(f"  positive  {int((oos_net > 0).sum())} of {len(oos_net)}")
print(f"\n  H0: mean = 0   ->   t = {t_stat:.3f},  p = {p_val:.3f}  (n = {len(oos_net)})")


In [ ]:
r_p, p_p = stats.pearsonr(ins, oos_net)
r_s, p_s = stats.spearmanr(ins, oos_net)
print("Does in-sample performance predict out-of-sample performance?")
print(f"  pearson  r   = {r_p:.3f}  (p = {p_p:.3f})")
print(f"  spearman rho = {r_s:.3f}  (p = {p_s:.3f})")

fig, ax = plt.subplots(figsize=(6, 5))
ax.axhline(0, lw=0.8, color="0.7")
ax.axvline(0, lw=0.8, color="0.7")
ax.scatter(ins, oos_net, s=45, zorder=3)
for name in net_ret.index:
    ax.annotate(name, (ins[name], oos_net[name]), fontsize=8,
                xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("in-sample net total return %")
ax.set_ylabel("out-of-sample net total return %")
ax.set_title("In-sample vs out-of-sample, 10 pairs")
plt.show()


**The mean out-of-sample return is statistically indistinguishable from zero**
(t = 0.08, p = 0.94), with a standard deviation of **26 percentage points** around it. The
median is *negative* while the mean is slightly positive, because one pair (UNP/CSX,
+55.9%) carries the average on its own.

This is a different and more defensible finding than the one the six-pair version of this
study reported. It is not "pairs trading loses money". It is:

> **Across a universe of economically-linked pairs, the expected out-of-sample return of
> this strategy is indistinguishable from zero, and the variance across pairs is so large
> that any three-pair study — including this one, at its earlier six-pair stage — can
> report whichever conclusion its pair selection happens to produce.**

The scatter shows a positive in-sample/out-of-sample relationship that sits right on
the edge of significance: **Spearman ρ = 0.64 (p = 0.048)** just clears the 5% level, while
**Pearson r = 0.60 (p = 0.069)** just misses it. A result that changes verdict depending on
whether you rank the data first, at n = 10, should not be reported as a finding in either
direction. Read honestly it says: in-sample performance is probably not pure noise as a
predictor of out-of-sample performance, but this test lacks the power to establish that, and
it is nowhere near reliable enough to select pairs on.

**Why the flat universe matters here specifically.** Had this study kept a headline tier
and an appendix tier, the three original headline pairs (WM/RSG, FOXA/FOX, SPY/VOO) would
have shown 0 of 3 profitable and the conclusion would have read as a clean negative
result. The four pairs added last would have shown 3 of 4 profitable. Same method, same
parameters, same period — opposite headlines, decided entirely by which pairs sat in which
tier. Removing the tiers is what makes that impossible to do accidentally.


## 6. Full metrics

Every pair, both periods, both bases.


In [ ]:
cols = ["total_return", "annualized_return", "sharpe", "max_drawdown",
        "turnover", "hit_rate", "n_trades", "n_periods"]

full = metrics[metrics.period != "full"].set_index(["pair", "period", "basis"])[cols]
full.reindex(
    pd.MultiIndex.from_product(
        [PAIRS, ["in_sample", "out_of_sample"], ["gross", "net"]],
        names=["pair", "period", "basis"],
    )
).round(4)


Two caveats visible in this table.

**Hit rates sit between 0.45 and 0.57**, computed over active days only. Even UNP/CSX, the
best performer, wins on only 56% of its active days — its return comes from a modest edge
applied across 68 trades, not from a handful of spectacular ones.

**FOXA/FOX has 709 in-sample days against 1,763 for every other pair.** FOX only began
trading on 2019-03-13, after the Disney transaction, so its hedge ratio is fit on under
half the data the others get and all of its statistics are correspondingly noisier. This is
a limitation of the pair, not something to correct for.


## 7. Equity curves and drawdowns

Regenerated live from `pairs_teardown.study.run_study` — the same code path `make run`
uses, so these cannot drift from the tables above. The dashed line marks the IS/OOS
boundary: everything to its right was scored once, with the hedge ratio and all parameters
already frozen.

On each equity chart the **gap between the gross and net lines is the transaction-cost
wedge**.


In [ ]:
prices_raw = load_or_download(
    list(cfg.tickers), cfg.data.start, cfg.data.end, ROOT / cfg.data.cache_dir
)
runs = {r.pair.name: r for r in run_study(prices_raw, cfg)}
split_date = pd.Timestamp(cfg.split.in_sample_end)

# Ordered best to worst out-of-sample, so the dispersion of §5 is visible as you scroll.
for name in headline.index:
    r = runs[name]
    gross_equity = (1 + r.result.gross_returns).cumprod()

    fig = plot_equity_curve(
        r.result.equity_curve, gross_equity, title=f"{name} — equity (gross vs net)"
    )
    fig.axes[0].axvline(split_date, ls="--", lw=1, color="0.4")
    fig.axes[0].annotate(" OOS →", (split_date, fig.axes[0].get_ylim()[1]),
                         va="top", fontsize=9, color="0.3")
    plt.show()

    fig = plot_drawdown(r.result.equity_curve, title=f"{name} — net drawdown")
    fig.axes[0].axvline(split_date, ls="--", lw=1, color="0.4")
    plt.show()
    plt.close("all")


## 8. What this notebook does and does not establish

**Establishes.** With window, entry and exit fixed a priori, the sizing hedge ratio fit
in-sample only, and 6 bps per side of cost: across ten economically-linked pairs the mean
out-of-sample return is indistinguishable from zero (p = 0.94) with a 26pp cross-sectional
standard deviation; costs consume roughly 80% of the mean gross return and flip the sign of
two pairs; and the cost drag is quantitatively accounted for (§4).

**Does not establish.** That pairs trading works, or that it does not. Ten pairs is far too
small a cross-section to resolve a mean this close to zero against a standard deviation
this large — that is the point of §5, not a limitation of it. Also untested: intraday
execution, dynamic position sizing, a genuinely large universe, and cost models other than a
flat retail-taker assumption.

**Does not check.** How much of any of this depends on the frozen parameter choices. That
is `04_sensitivity_analysis.ipynb`'s job, and reading this notebook without it will
overstate how solid these numbers are.

**Not corrected for.** Survivorship — all 20 tickers were selected in 2026 from companies
that still exist and still trade. A pair that de-listed or was acquired mid-sample never
entered the universe, which biases these results *upward*. See `05_writeup.ipynb` §2.2.
